In [1]:
import pandas as pd
import scanpy as sc

In [2]:
# 读取 .h5ad 文件
adata_all = sc.read("/public/users/xueyupeng/VZV/lvzhu/record/adata_all_celltype_50k.h5ad")
print(adata_all)

AnnData object with n_obs × n_vars = 7783039 × 52
    obs: 'ID', 'Visit', 'Sample', 'batch', 'leiden', 'celltype', 'Timepoint'
    uns: 'Visit_colors', 'celltype_colors', 'leiden', 'leiden_colors', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'raw'
    obsp: 'connectivities', 'distances'


In [3]:
# 2. 定义细胞类型注释文件的路径
celltype_files = {
    'CD4T': '/public/users/xueyupeng/VZV/lvzhu/record/subtype_annotations_CD4T.csv',
    'CD8T': '/public/users/xueyupeng/VZV/lvzhu/record/subtype_annotations_CD8.csv',
    'B': '/public/users/xueyupeng/VZV/lvzhu/record/subtype_annotations_B.csv',
    'NK': '/public/users/xueyupeng/VZV/lvzhu/record/subtype_annotations_NK.csv',
    'Mono': '/public/users/xueyupeng/VZV/lvzhu/record/subtype_annotations_Mono.csv',
    'DC': '/public/users/xueyupeng/VZV/lvzhu/record/subtype_annotations_DC.csv',
    'Plasma': '/public/users/xueyupeng/VZV/lvzhu/record/subtype_annotations_Plasma.csv',
}

# 3. 读取各细胞类型的亚群注释信息并合并
all_annotations = pd.DataFrame()

for celltype, file_path in celltype_files.items():
    # 读取每个细胞类型的亚群注释信息
    celltype_annotations = pd.read_csv(file_path)
    
    # 确保cellid列存在并与adata_all.obs进行合并
    celltype_annotations = celltype_annotations[['cellid', 'subtype']]  # 保留cellid和subtype列
    celltype_annotations.rename(columns={'subtype': f'{celltype}_subtype'}, inplace=True)  # 给subtype列重新命名
    
    # 合并到all_annotations DataFrame
    if all_annotations.empty:
        all_annotations = celltype_annotations
    else:
        all_annotations = pd.merge(all_annotations, celltype_annotations, on='cellid', how='outer')

In [4]:
all_annotations.head()

,cellid,CD4T_subtype,CD8T_subtype,B_subtype,NK_subtype,Mono_subtype,DC_subtype,Plasma_subtype
0,0,CD4TEM,NaN,NaN,NaN,NaN,NaN,NaN
1,0-1,NaN,CD8TEM,NaN,NaN,NaN,NaN,NaN
2,0-10,CD4TEM,NaN,NaN,NaN,NaN,NaN,NaN
3,0-11,CD4TEM,NaN,NaN,NaN,NaN,NaN,NaN
4,0-12,CD4TEM,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df = pd.DataFrame(all_annotations)

# 将 'NaN' 字符串转换为实际的 NaN 值
df = df.replace('NaN', pd.NA)

# 统计每一行的 NaN 个数（忽略 cellid 列）
na_count_per_row = df.drop(columns='cellid').isna().sum(axis=1)

In [6]:
na_count_per_row.value_counts()

6    7783039
Name: count, dtype: int64

In [7]:
df = pd.DataFrame(all_annotations)

# 使用 any() 检查 cellid 列中是否有重复值
has_duplicates = df['cellid'].duplicated().any()

# 输出是否存在重复值
if has_duplicates:
    print("cellid 列中有重复值")
else:
    print("cellid 列中没有重复值")

cellid 列中没有重复值


In [8]:
import pandas as pd

# 假设你的数据已经是一个 DataFrame，叫做 `subtype_annotations`

# 1. 使用 melt 方法将多列转为长格式，去掉 NaN 值
subtype_annotations_melted = all_annotations.melt(id_vars=['cellid'], 
                                                      value_vars=['CD4T_subtype', 'CD8T_subtype', 'B_subtype', 
                                                                  'NK_subtype', 'Mono_subtype', 'DC_subtype', 'Plasma_subtype'], 
                                                      var_name='cell_type', value_name='subtype')

# 2. 去除 NaN 值
subtype_annotations_melted = subtype_annotations_melted.dropna(subset=['subtype'])

# 3. 合并成单列
subtype_annotations_melted = subtype_annotations_melted.groupby('cellid')['subtype'].apply(lambda x: ', '.join(x)).reset_index()

# 打印结果
print(subtype_annotations_melted)

          cellid     subtype
0              0      CD4TEM
1            0-1      CD8TEM
2           0-10      CD4TEM
3           0-11      CD4TEM
4           0-12      CD4TEM
...          ...         ...
7783034  99999-5    NK_CD57-
7783035  99999-6   NK_CD11c+
7783036  99999-7  Naive CD4T
7783037  99999-8    NK_CD57-
7783038  99999-9     Naive B

[7783039 rows x 2 columns]


In [9]:
adata_all.obs['cellid'] = adata_all.obs.index

In [10]:
adata_all.obs

,ID,Visit,Sample,batch,leiden,celltype,Timepoint,cellid
243005,A004,V1,A004_V1,A004_V1,0,NK,V1Day1,243005
77917,A004,V1,A004_V1,A004_V1,0,NK,V1Day1,77917
212179,A004,V1,A004_V1,A004_V1,5,CD8T,V1Day1,212179
210568,A004,V1,A004_V1,A004_V1,10,CD4T,V1Day1,210568
144145,A004,V1,A004_V1,A004_V1,8,CD4T,V1Day1,144145
...,...,...,...,...,...,...,...,...
115479-27,A030,V4,A030_V4,A030_V4,0,NK,V2Day1,115479-27
212169-24,A030,V4,A030_V4,A030_V4,2,DC,V2Day1,212169-24
176143-29,A030,V4,A030_V4,A030_V4,0,NK,V2Day1,176143-29
175441-21,A030,V4,A030_V4,A030_V4,9,CD4T,V2Day1,175441-21


In [11]:
subtype_annotations_melted

,cellid,subtype
0,0,CD4TEM
1,0-1,CD8TEM
2,0-10,CD4TEM
3,0-11,CD4TEM
4,0-12,CD4TEM
...,...,...
7783034,99999-5,NK_CD57-
7783035,99999-6,NK_CD11c+
7783036,99999-7,Naive CD4T
7783037,99999-8,NK_CD57-


In [12]:
# 1. 获取adata_all.obs和subtype_annotations_melted中的cellid
adata_cellids = adata_all.obs['cellid'].astype(str).unique()
subtype_cellids = subtype_annotations_melted['cellid'].astype(str).unique()

# 2. 检查adata_all.obs中的cellid是否都在subtype_annotations_melted中
missing_in_subtype = set(adata_cellids) - set(subtype_cellids)
missing_in_adata = set(subtype_cellids) - set(adata_cellids)

# 输出缺失的cellid
print("在adata_all.obs中存在但在subtype_annotations_melted中缺失的cellid数量:", len(missing_in_subtype))
print("在subtype_annotations_melted中存在但在adata_all.obs中缺失的cellid数量:", len(missing_in_adata))

# 如果缺失的cellid存在，输出部分示例
if missing_in_subtype:
    print("\n缺失的cellid在adata_all.obs中但不在subtype_annotations_melted中:")
    print(list(missing_in_subtype)[:10])  # 输出前10个示例

if missing_in_adata:
    print("\n缺失的cellid在subtype_annotations_melted中但不在adata_all.obs中:")
    print(list(missing_in_adata)[:10])  # 输出前10个示例

在adata_all.obs中存在但在subtype_annotations_melted中缺失的cellid数量: 0
在subtype_annotations_melted中存在但在adata_all.obs中缺失的cellid数量: 0


In [13]:
adata_all.obs['cellid'] = adata_all.obs['cellid'].astype(str)
subtype_annotations_melted['cellid'] = subtype_annotations_melted['cellid'].astype(str)

In [14]:
# 4. 将合并后的注释信息添加到adata_all.obs中
import pandas as pd

# 在合并后手动恢复原来的 index
merged_data = pd.merge(
    adata_all.obs, 
    subtype_annotations_melted, 
    on='cellid', 
    how='left'
)

merged_data.index = adata_all.obs.index  # 恢复原有的 index

adata_all.obs = merged_data  # 将合并后的数据重新赋值给 adata_all.obs

In [15]:
adata_all

AnnData object with n_obs × n_vars = 7783039 × 52
    obs: 'ID', 'Visit', 'Sample', 'batch', 'leiden', 'celltype', 'Timepoint', 'cellid', 'subtype'
    uns: 'Visit_colors', 'celltype_colors', 'leiden', 'leiden_colors', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'raw'
    obsp: 'connectivities', 'distances'

In [16]:
adata_all.obs

,ID,Visit,Sample,batch,leiden,celltype,Timepoint,cellid,subtype
243005,A004,V1,A004_V1,A004_V1,0,NK,V1Day1,243005,NK_CD11c-
77917,A004,V1,A004_V1,A004_V1,0,NK,V1Day1,77917,NK_CD11c-
212179,A004,V1,A004_V1,A004_V1,5,CD8T,V1Day1,212179,CD8TCM
210568,A004,V1,A004_V1,A004_V1,10,CD4T,V1Day1,210568,Naive CD4T
144145,A004,V1,A004_V1,A004_V1,8,CD4T,V1Day1,144145,CD4TEM
...,...,...,...,...,...,...,...,...,...
115479-27,A030,V4,A030_V4,A030_V4,0,NK,V2Day1,115479-27,NK_CD57-
212169-24,A030,V4,A030_V4,A030_V4,2,DC,V2Day1,212169-24,cDC
176143-29,A030,V4,A030_V4,A030_V4,0,NK,V2Day1,176143-29,NK_CD11c-
175441-21,A030,V4,A030_V4,A030_V4,9,CD4T,V2Day1,175441-21,CD4CTL


In [17]:
# Step 1: 删除subtype为"unknown"的细胞数据
adata_all = adata_all[adata_all.obs['subtype'] != 'unknown'].copy()

# Step 2: 确认数据已更新
print(f"After removing 'unknown' subtypes, the dataset now has {adata_all.obs.shape[0]} cells.")

After removing 'unknown' subtypes, the dataset now has 7752210 cells.


In [18]:
adata_all.write('/public/users/xueyupeng/VZV/lvzhu/record/adata_all_subtypes.h5ad')

In [29]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Step 1: 确保sub_markers的标记物存在于adata_all.var_names中
sub_markers = ['IgD_142Nd', 'IFN-a_144Nd', 'CXCL9_146Nd', 'CD86_147Sm', 'H3K27ac_148Nd', 
               'IL-1b_149Sm', '4E-BP1_150Nd', 'pS6_151Eu', 'CCR2_152Sm', 'TNF-a_153Eu', 
               'IL-4_154Sm', 'CASP-8_156Gd', 'CD38_159Tb', 'CD123_161Dy', 'Ki-67_162Dy', 
               'CXCL10_163Dy', 'Arginase-1_164Dy', 'IL-6_165Ho', 'MCP-1_166Er', 
               'CD69_167Er', 'STC1_168Er', 'IFN-g_172Yb', 'pSTAT1_175Lu', 'pSTAT3_176Yb']

sub_markers = ['HLA-DR_171Yb']

# 检查sub_markers中的每个标记物是否在adata_all.var_names中
missing_markers = [marker for marker in sub_markers if marker not in adata_all.var_names]
if missing_markers:
    print(f"The following markers are missing from adata_all.var_names: {missing_markers}")
else:
    print("All markers are present in adata_all.var_names.")

# Step 2: 获取标记物的列索引
sub_marker_indices = [adata_all.var_names.get_loc(marker) for marker in sub_markers]

# Step 3: 使用这些索引提取adata_all.X的相关列数据，并转换为DataFrame
data_long = adata_all.obs[['Timepoint', 'subtype']].join(pd.DataFrame(adata_all.X[:, sub_marker_indices], 
                                                                      columns=sub_markers, 
                                                                      index=adata_all.obs.index))

All markers are present in adata_all.var_names.


In [30]:
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import pandas as pd
import numpy as np

# Step 1: 设置保存路径
output_dir = '/public/users/xueyupeng/VZV/lvzhu/figures_subtype_violinplot/'

# Step 2: 获取subtype的所有唯一值，定义颜色
subtypes = data_long['subtype'].unique()
num_subtypes = len(subtypes)

# 使用tab20调色板，保证每个subtype都有一个唯一的颜色
palette = sns.color_palette("tab20", num_subtypes)
subtype_palette = dict(zip(subtypes, palette))

# Step 2: 遍历每个 marker 绘制小提琴图并保存
for marker in sub_markers:
    # Step 3: 过滤数据，选择当前 marker
    data_marker = data_long[['subtype', 'Timepoint', marker]]

    # Step 4: 绘制小提琴图
    plt.figure(figsize=(16, 8))
    # 绘制小提琴图，按照Timepoint区分，并根据subtype进行分组
    sns.violinplot(x="Timepoint", y=marker, hue="subtype", data=data_long,
                   inner="quart", palette=subtype_palette, split=True)
    # Step 5: 设置标题
    plt.title(f'{marker} Violin Plot by Subtype and Timepoint')

    # Step 6: 调整图例
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))

    # Step 7: 保存每个小提琴图
    output_path = f'{output_dir}{marker}_violinplot.png'
    plt.savefig(output_path, bbox_inches='tight', dpi=300)

    # Step 8: 清空当前图形，准备下一张
    plt.close()


In [21]:
import scanpy as sc
import matplotlib.pyplot as plt

# Create a new figure with specific size and set the DPI
fig, ax = plt.subplots(figsize=(10, 8), dpi=300)

# Generate UMAP plot (without using show=True directly)
sc.pl.umap(adata_all, 
           color='subtype', 
           palette=subtype_palette, 
           title='UMAP of Subtypes', 
           size=0.2, 
           legend_loc='right margin', 
           ax=ax,  # Use the explicitly created axes for the plot
           show=False)

# Save the plot to the desired directory with DPI set to 300
plt.savefig('/public/users/xueyupeng/VZV/lvzhu/figures_subtype_violinplot/umap_subtype.png', dpi=300, bbox_inches='tight')

plt.close()  # Close the plot to avoid showing it again

In [22]:
import pandas as pd
import os

# ========= 输入数据 =========
df = adata_all.obs.copy()

# ========= 基础清洗 =========
df = df.dropna(subset=["Sample", "subtype"])

# ========= 1️⃣ 计算每个 Sample × subtype 的计数 =========
cell_counts = (
    df.groupby(["Sample", "subtype"])
    .size()
    .reset_index(name="Count")
)

# ========= 2️⃣ 计算每个 Sample 的总细胞数 =========
total_counts = (
    df.groupby("Sample")
    .size()
    .reset_index(name="Total")
)

# ========= 3️⃣ 提取 Sample 级别的元信息 =========
meta = (
    df[["Sample", "ID", "Visit", "Timepoint"]]
    .drop_duplicates()
)

# ========= 4️⃣ 合并 =========
merged = pd.merge(cell_counts, total_counts, on="Sample")
merged = pd.merge(merged, meta, on="Sample", how="left")

# ========= 5️⃣ 计算比例 =========
merged["Proportion"] = merged["Count"] / merged["Total"]
merged["Proportion_%"] = merged["Proportion"] * 100

# ========= 6️⃣ 透视表 =========
pivot_prop = merged.pivot_table(
    index="Sample",
    columns="subtype",
    values="Proportion",
    fill_value=0
)

pivot_count = merged.pivot_table(
    index="Sample",
    columns="subtype",
    values="Count",
    fill_value=0
)

# ========= 7️⃣ 保存 =========
save_path = "/public/users/xueyupeng/VZV/lvzhu/record/subtype_proportion.xlsx"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

with pd.ExcelWriter(save_path) as writer:
    merged.to_excel(writer, sheet_name="long_format", index=False)
    pivot_prop.to_excel(writer, sheet_name="proportion_pivot")
    pivot_count.to_excel(writer, sheet_name="count_pivot")

print(f"✓ 保存完成: {save_path}")
print(f"✓ merged shape: {merged.shape}")

/tmp/ipykernel_2301368/4169745778.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(["Sample", "subtype"])
/tmp/ipykernel_2301368/4169745778.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("Sample")
/tmp/ipykernel_2301368/4169745778.py:39: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_prop = merged.pivot_table(
/tmp/ipykernel_2301368/4169745778.py:46: FutureWarning: The default value of observed=False is deprecat

✓ 保存完成: /public/users/xueyupeng/VZV/lvzhu/record/subtype_proportion.xlsx
✓ merged shape: (3276, 9)


In [23]:
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

# ========= 路径 =========
input_path = "/public/users/xueyupeng/VZV/lvzhu/record/subtype_proportion.xlsx"
save_dir = "/public/users/xueyupeng/VZV/lvzhu/subtype_time_boxplot/"
os.makedirs(save_dir, exist_ok=True)

# ========= 读取数据 =========
df = pd.read_excel(input_path, sheet_name="long_format")

df = df[["Sample", "subtype", "Proportion", "ID", "Timepoint"]].dropna()

# ========= Timepoint 顺序 =========
time_order = ["V1Day0", "V1Day1", "V2Day0", "V2Day1"]
df["Timepoint"] = pd.Categorical(df["Timepoint"], categories=time_order, ordered=True)

# ========= 比较组 =========
comparisons = [
    ("V1Day0", "V1Day1"),
    ("V2Day0", "V2Day1"),
    ("V1Day0", "V2Day1")
]

# ========= 显著性标注 =========
def add_sig(ax, x1, x2, y, p):
    if p < 0.001:
        text = "***"
    elif p < 0.01:
        text = "**"
    elif p < 0.05:
        text = "*"
    else:
        text = "ns"

    ax.plot([x1, x1, x2, x2], [y, y*1.01, y*1.01, y], color="black", lw=1.5)
    ax.text((x1+x2)/2, y*1.02, text, ha="center", color="black")

# ========= 主循环 =========
subtypes = df["subtype"].unique()

for ct in subtypes:

    sub = df[df["subtype"] == ct].copy()

    plt.figure(figsize=(8,6))

    # ===== 箱线图（黑色）=====
    ax = sns.boxplot(
        data=sub,
        x="Timepoint",
        y="Proportion",
        order=time_order,
        color="white",              # 填充白色
        boxprops=dict(edgecolor="black"),
        medianprops=dict(color="black"),
        whiskerprops=dict(color="black"),
        capprops=dict(color="black"),
        showfliers=False
    )

    # ===== 散点（黑色）=====
    sns.stripplot(
        data=sub,
        x="Timepoint",
        y="Proportion",
        order=time_order,
        color="black",
        size=4,
        alpha=0.8,
        jitter=True
    )

    plt.title(f"{ct} proportion over time")
    plt.ylabel("Proportion", color="black")
    plt.xlabel("Timepoint", color="black")

    # 坐标轴字体黑色
    ax.tick_params(axis='x', colors='black')
    ax.tick_params(axis='y', colors='black')

    # 去掉 legend（或也可设为黑色）
    plt.legend([], [], frameon=False)

    # ========= 统计 =========
    y_max = sub["Proportion"].max()
    y_base = y_max * 1.01
    step = y_max * 0.05

    for i, (tp1, tp2) in enumerate(comparisons):

        df1 = sub[sub["Timepoint"] == tp1]
        df2 = sub[sub["Timepoint"] == tp2]

        if df1.empty or df2.empty:
            continue

        common_ids = set(df1["ID"]).intersection(set(df2["ID"]))

        if len(common_ids) < 3:
            continue

        df1_id = df1[df1["ID"].isin(common_ids)].set_index("ID")
        df2_id = df2[df2["ID"].isin(common_ids)].set_index("ID")

        df1_id = df1_id.sort_index()
        df2_id = df2_id.sort_index()

        x = df1_id["Proportion"]
        y = df2_id["Proportion"]

        common = x.index.intersection(y.index)

        if len(common) < 3:
            continue

        x = x.loc[common]
        y = y.loc[common]

        try:
            stat, p = wilcoxon(x, y)
        except:
            continue

        x1 = time_order.index(tp1)
        x2 = time_order.index(tp2)
        y_pos = y_base + i * step

        add_sig(ax, x1, x2, y_pos, p)

    plt.tight_layout()

    save_path = os.path.join(save_dir, f"{ct}.png")
    plt.savefig(save_path, dpi=300)
    plt.close()

print(f"✓ 所有图已保存到: {save_dir}")

✓ 所有图已保存到: /public/users/xueyupeng/VZV/lvzhu/subtype_time_boxplot/


In [31]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Step 1: 确保sub_markers的标记物存在于adata_all.var_names中
sub_markers = ['IgD_142Nd', 'IFN-a_144Nd', 'CXCL9_146Nd', 'CD86_147Sm', 'H3K27ac_148Nd', 
               'IL-1b_149Sm', '4E-BP1_150Nd', 'pS6_151Eu', 'CCR2_152Sm', 'TNF-a_153Eu', 
               'IL-4_154Sm', 'CASP-8_156Gd', 'CD38_159Tb', 'CD123_161Dy', 'Ki-67_162Dy', 
               'CXCL10_163Dy', 'Arginase-1_164Dy', 'IL-6_165Ho', 'MCP-1_166Er', 
               'CD69_167Er', 'STC1_168Er', 'IFN-g_172Yb', 'pSTAT1_175Lu', 'pSTAT3_176Yb']
sub_markers = ['HLA-DR_171Yb']
# Step 2: 获取标记物的列索引
sub_marker_indices = [adata_all.var_names.get_loc(marker) for marker in sub_markers]

# Step 3: 使用这些索引提取adata_all.X的相关列数据，并转换为DataFrame
data_long = adata_all.obs[['Timepoint', 'celltype']].join(pd.DataFrame(adata_all.X[:, sub_marker_indices], 
                                                                      columns=sub_markers, 
                                                                      index=adata_all.obs.index))

In [32]:
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import pandas as pd
import numpy as np

# Step 1: 设置保存路径
output_dir = '/public/users/xueyupeng/VZV/lvzhu/figures_celltype_violinplot/'

# Step 2: 获取celltype的所有唯一值，定义颜色
celltypes = data_long['celltype'].unique()
num_celltypes = len(celltypes)

# 使用tab20调色板，保证每个celltype都有一个唯一的颜色
palette = sns.color_palette("tab20", num_celltypes)
celltype_palette = dict(zip(celltypes, palette))

# Step 2: 遍历每个 marker 绘制小提琴图并保存
for marker in sub_markers:
    # Step 3: 过滤数据，选择当前 marker
    data_marker = data_long[['celltype', 'Timepoint', marker]]

    # Step 4: 绘制小提琴图
    plt.figure(figsize=(16, 8))
    # 绘制小提琴图，按照Timepoint区分，并根据celltype进行分组
    sns.violinplot(x="Timepoint", y=marker, hue="celltype", data=data_long,
                   inner="quart", palette=celltype_palette, split=True)
    # Step 5: 设置标题
    plt.title(f'{marker} Violin Plot by celltype and Timepoint')

    # Step 6: 调整图例
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))

    # Step 7: 保存每个小提琴图
    output_path = f'{output_dir}{marker}_violinplot.png'
    plt.savefig(output_path, bbox_inches='tight', dpi=300)

    # Step 8: 清空当前图形，准备下一张
    plt.close()

In [26]:
import scanpy as sc
import matplotlib.pyplot as plt

# Create a new figure with specific size and set the DPI
fig, ax = plt.subplots(figsize=(10, 8), dpi=300)

# Generate UMAP plot (without using show=True directly)
sc.pl.umap(adata_all, 
           color='celltype', 
           palette=celltype_palette, 
           title='UMAP of celltypes', 
           size=0.2, 
           legend_loc='right margin', 
           ax=ax,  # Use the explicitly created axes for the plot
           show=False)

# Save the plot to the desired directory with DPI set to 300
plt.savefig('/public/users/xueyupeng/VZV/lvzhu/figures_celltype_violinplot/umap_celltype.png', dpi=300, bbox_inches='tight')

plt.close()  # Close the plot to avoid showing it again

In [27]:
print(adata_all.obs.subtype.unique())

['NK_CD11c-', 'CD8TCM', 'Naive CD4T', 'CD4TEM', 'cMono', ..., 'pDC', 'NK_Ki67+', 'Unswitch memory B', 'Plasma', 'CD8TCM_Ki-67+']
Length: 21
Categories (21, object): ['BM_CD27+', 'BM_CD27-', 'CD4CTL', 'CD4TCM', ..., 'Unswitch memory B', 'cDC', 'cMono', 'pDC']


In [28]:
print(list(adata_all.obs.subtype.unique()))

['NK_CD11c-', 'CD8TCM', 'Naive CD4T', 'CD4TEM', 'cMono', 'NK_CD11c+', 'NKbright', 'Naive B', 'cDC', 'NK_CD57-', 'CD8TEM', 'CD4CTL', 'BM_CD27-', 'CD8CTL', 'BM_CD27+', 'CD4TCM', 'pDC', 'NK_Ki67+', 'Unswitch memory B', 'Plasma', 'CD8TCM_Ki-67+']


In [33]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats

# Step 1: 确保sub_markers的标记物存在于adata_all.var_names中
sub_markers = ['IgD_142Nd', 'IFN-a_144Nd', 'CXCL9_146Nd', 'CD86_147Sm', 'H3K27ac_148Nd', 
               'IL-1b_149Sm', '4E-BP1_150Nd', 'pS6_151Eu', 'CCR2_152Sm', 'TNF-a_153Eu', 
               'IL-4_154Sm', 'CASP-8_156Gd', 'CD38_159Tb', 'CD123_161Dy', 'Ki-67_162Dy', 
               'CXCL10_163Dy', 'Arginase-1_164Dy', 'IL-6_165Ho', 'MCP-1_166Er', 
               'CD69_167Er', 'STC1_168Er', 'IFN-g_172Yb', 'pSTAT1_175Lu', 'pSTAT3_176Yb','HLA-DR_171Yb']

# 检查sub_markers中的每个标记物是否在adata_all.var_names中
missing_markers = [marker for marker in sub_markers if marker not in adata_all.var_names]
if missing_markers:
    print(f"The following markers are missing from adata_all.var_names: {missing_markers}")
else:
    print("All markers are present in adata_all.var_names.")

# Step 2: 获取标记物的列索引
sub_marker_indices = [adata_all.var_names.get_loc(marker) for marker in sub_markers]

# Step 3: 使用这些索引提取adata_all.X的相关列数据，并转换为DataFrame
data_long = adata_all.obs[['Timepoint', 'subtype']].join(pd.DataFrame(adata_all.X[:, sub_marker_indices], 
                                                                      columns=sub_markers, 
                                                                      index=adata_all.obs.index))

# Step 4: 设置保存路径
output_dir = '/public/users/xueyupeng/VZV/lvzhu/figures_subtype_violinplot_comp/'

# Step 5: 获取subtype的所有唯一值，定义颜色
subtypes = data_long['subtype'].unique()
num_subtypes = len(subtypes)

# 使用tab20调色板，保证每个subtype都有一个唯一的颜色
palette = sns.color_palette("tab20", num_subtypes)
subtype_palette = dict(zip(subtypes, palette))

All markers are present in adata_all.var_names.


In [55]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats

# Step 1: 确保sub_markers的标记物存在于adata_all.var_names中
sub_markers = ['IgD_142Nd', 'IFN-a_144Nd', 'CXCL9_146Nd', 'CD86_147Sm', 'H3K27ac_148Nd', 
               'IL-1b_149Sm', '4E-BP1_150Nd', 'pS6_151Eu', 'CCR2_152Sm', 'TNF-a_153Eu', 
               'IL-4_154Sm', 'CASP-8_156Gd', 'CD38_159Tb', 'CD123_161Dy', 'Ki-67_162Dy', 
               'CXCL10_163Dy', 'Arginase-1_164Dy', 'IL-6_165Ho', 'MCP-1_166Er', 
               'CD69_167Er', 'STC1_168Er', 'IFN-g_172Yb', 'pSTAT1_175Lu', 'pSTAT3_176Yb','HLA-DR_171Yb']  # 你可以继续加入其他markers

# 检查sub_markers中的每个标记物是否在adata_all.var_names中
missing_markers = [marker for marker in sub_markers if marker not in adata_all.var_names]
if missing_markers:
    print(f"The following markers are missing from adata_all.var_names: {missing_markers}")
else:
    print("All markers are present in adata_all.var_names.")

# Step 2: 获取标记物的列索引
sub_marker_indices = [adata_all.var_names.get_loc(marker) for marker in sub_markers]

# Step 3: 使用这些索引提取adata_all.X的相关列数据，并转换为DataFrame
data_long = adata_all.obs[['Timepoint', 'subtype']].join(pd.DataFrame(adata_all.X[:, sub_marker_indices], 
                                                                      columns=sub_markers, 
                                                                      index=adata_all.obs.index))

# Step 4: 设置保存路径
output_dir = '/public/users/xueyupeng/VZV/lvzhu/figures_subtype_violinplot_comp/'

# Step 5: 获取subtype的所有唯一值，定义颜色
subtypes = data_long['subtype'].unique()
num_subtypes = len(subtypes)

# 使用tab20调色板，保证每个subtype都有一个唯一的颜色
subtype_palette = sns.color_palette("tab20", num_subtypes)
subtype_palette = dict(zip(subtypes, subtype_palette))

# Step 6: 获取Timepoint的所有唯一值并定义颜色
timepoints = data_long['Timepoint'].unique()
timepoint_palette = sns.color_palette("Set2", len(timepoints))  # 使用Set2调色板为时间点分配颜色
timepoint_palette = dict(zip(timepoints, timepoint_palette))

# Step 7: 遍历每个 marker 绘制小提琴图并保存
for marker in sub_markers:
    # Step 8: 过滤数据，选择当前 marker
    data_marker = data_long[['subtype', 'Timepoint', marker]]

    # Step 9: 为显著性检验创建两个比较组
    # 比较时间点：V1Day0与V1Day1、V2Day0与V2Day1
    comparison_groups = [
        (('V1Day0', 'V1Day1'), 'V1Day0 vs V1Day1'),
        (('V2Day0', 'V2Day1'), 'V2Day0 vs V2Day1')
    ]

    # Step 10: 绘制小提琴图
    plt.figure(figsize=(14, 6))
    sns.violinplot(x="subtype", y=marker, hue="Timepoint", data=data_marker, 
                   inner="quart", palette=timepoint_palette, split=True)
    
    # Step 11: 设置标题
    plt.title(f'{marker} Violin Plot by Subtype and Timepoint')

    # Step 12: 调整图例
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))

    # Step 13: 进行显著性检验并标出显著性标签
    for (timepoints, comparison_label) in comparison_groups:
        # 过滤出指定时间点的数据
        timepoint_data = data_marker[data_marker['Timepoint'].isin(timepoints)]
        # 进行 Mann-Whitney U test 或 t-test 检验
        p_values = {}
        for subtype in subtypes:
            subtype_data = timepoint_data[timepoint_data['subtype'] == subtype]
            group1 = subtype_data[subtype_data['Timepoint'] == timepoints[0]][marker]
            group2 = subtype_data[subtype_data['Timepoint'] == timepoints[1]][marker]
            
            # 使用 Mann-Whitney U test（对于非正态分布的数据）
            stat, p_val = stats.ttest_ind(group1, group2, equal_var=False)  # equal_var=False表示Welch's t-test，不假设方差相等
            p_values[subtype] = p_val
        # 标出显著性标签（只显示 * 符号）
        for subtype, p_val in p_values.items():
            if p_val < 0.001:
                # 动态设置y坐标，避免与小提琴图重叠
                max_y = data_marker[marker].max()  # 获取当前marker的最大值
                if timepoints == ('V1Day0', 'V1Day1'):
                    # 对于V1Day0与V1Day1比较，标签偏左
                    x_position = subtypes.tolist().index(subtype) - 0.2
                elif timepoints == ('V2Day0', 'V2Day1'):
                    # 对于V2Day0与V2Day1比较，标签偏右
                    x_position = subtypes.tolist().index(subtype) + 0.2
                # 在图中添加显著性标记
                plt.text(x_position, max_y + 0.5, "***", horizontalalignment='center', 
                         verticalalignment='center', fontsize=8, color='black')
            elif  p_val < 0.01:
                # 动态设置y坐标，避免与小提琴图重叠
                max_y = data_marker[marker].max()  # 获取当前marker的最大值
                if timepoints == ('V1Day0', 'V1Day1'):
                    # 对于V1Day0与V1Day1比较，标签偏左
                    x_position = subtypes.tolist().index(subtype) - 0.2
                elif timepoints == ('V2Day0', 'V2Day1'):
                    # 对于V2Day0与V2Day1比较，标签偏右
                    x_position = subtypes.tolist().index(subtype) + 0.2
                # 在图中添加显著性标记
                plt.text(x_position, max_y + 0.5, "**", horizontalalignment='center', 
                         verticalalignment='center', fontsize=8, color='black')
            elif  p_val < 0.05:
                # 动态设置y坐标，避免与小提琴图重叠
                max_y = data_marker[marker].max()  # 获取当前marker的最大值
                if timepoints == ('V1Day0', 'V1Day1'):
                    # 对于V1Day0与V1Day1比较，标签偏左
                    x_position = subtypes.tolist().index(subtype) - 0.2
                elif timepoints == ('V2Day0', 'V2Day1'):
                    # 对于V2Day0与V2Day1比较，标签偏右
                    x_position = subtypes.tolist().index(subtype) + 0.2
                # 在图中添加显著性标记
                plt.text(x_position, max_y + 0.5, "*", horizontalalignment='center', 
                         verticalalignment='center', fontsize=8, color='black')
    # Step 14: 旋转x轴标签 45°并调整其位置
    plt.xticks(rotation=45, ha='right')
    # Step 15: 隐藏x轴标题
    plt.xlabel('')
    # Step 14: 保存每个小提琴图
    output_path = f'{output_dir}{marker}_violinplot.png'
    plt.savefig(output_path, bbox_inches='tight', dpi=300)

    # Step 15: 清空当前图形，准备下一张
    plt.close()

All markers are present in adata_all.var_names.


In [57]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.stats.multitest as smm

# Step 1: 确保sub_markers的标记物存在于adata_all.var_names中
sub_markers = ['IgD_142Nd', 'IFN-a_144Nd', 'CXCL9_146Nd', 'CD86_147Sm', 'H3K27ac_148Nd', 
               'IL-1b_149Sm', '4E-BP1_150Nd', 'pS6_151Eu', 'CCR2_152Sm', 'TNF-a_153Eu', 
               'IL-4_154Sm', 'CASP-8_156Gd', 'CD38_159Tb', 'CD123_161Dy', 'Ki-67_162Dy', 
               'CXCL10_163Dy', 'Arginase-1_164Dy', 'IL-6_165Ho', 'MCP-1_166Er', 
               'CD69_167Er', 'STC1_168Er', 'IFN-g_172Yb', 'pSTAT1_175Lu', 'pSTAT3_176Yb','HLA-DR_171Yb']  # 你可以继续加入其他markers

# 检查sub_markers中的每个标记物是否在adata_all.var_names中
missing_markers = [marker for marker in sub_markers if marker not in adata_all.var_names]
if missing_markers:
    print(f"The following markers are missing from adata_all.var_names: {missing_markers}")
else:
    print("All markers are present in adata_all.var_names.")

# Step 2: 获取标记物的列索引
sub_marker_indices = [adata_all.var_names.get_loc(marker) for marker in sub_markers]

# Step 3: 使用这些索引提取adata_all.X的相关列数据，并转换为DataFrame
data_long = adata_all.obs[['Timepoint', 'subtype']].join(pd.DataFrame(adata_all.X[:, sub_marker_indices], 
                                                                      columns=sub_markers, 
                                                                      index=adata_all.obs.index))

# Step 4: 设置保存路径
output_dir = '/public/users/xueyupeng/VZV/lvzhu/figures_subtype_violinplot_comp/'

# Step 5: 获取subtype的所有唯一值，定义颜色
subtypes = data_long['subtype'].unique()
num_subtypes = len(subtypes)

# 使用tab20调色板，保证每个subtype都有一个唯一的颜色
subtype_palette = sns.color_palette("tab20", num_subtypes)
subtype_palette = dict(zip(subtypes, subtype_palette))

# Step 6: 获取Timepoint的所有唯一值并定义颜色
timepoints = data_long['Timepoint'].unique()
timepoint_palette = sns.color_palette("Set2", len(timepoints))  # 使用Set2调色板为时间点分配颜色
timepoint_palette = dict(zip(timepoints, timepoint_palette))

# Step 7: 遍历每个 marker 绘制小提琴图并保存
for marker in sub_markers:
    # Step 8: 过滤数据，选择当前 marker
    data_marker = data_long[['subtype', 'Timepoint', marker]]

    # Step 9: 为显著性检验创建两个比较组
    # 比较时间点：V1Day0与V1Day1、V2Day0与V2Day1
    comparison_groups = [
        (('V1Day0', 'V1Day1'), 'V1Day0 vs V1Day1'),
        (('V2Day0', 'V2Day1'), 'V2Day0 vs V2Day1')
    ]

    # Step 10: 绘制小提琴图
    plt.figure(figsize=(14, 6))
    sns.violinplot(x="subtype", y=marker, hue="Timepoint", data=data_marker, 
                   inner="quart", palette=timepoint_palette, split=True)
    
    # Step 11: 设置标题
    plt.title(f'{marker} Violin Plot by Subtype and Timepoint')

    # Step 12: 调整图例
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))

    # Step 13: 进行显著性检验并标出显著性标签
    for (timepoints, comparison_label) in comparison_groups:
        # 过滤出指定时间点的数据
        timepoint_data = data_marker[data_marker['Timepoint'].isin(timepoints)]
        # 进行 Mann-Whitney U test 或 t-test 检验
        p_values = {}
        for subtype in subtypes:
            subtype_data = timepoint_data[timepoint_data['subtype'] == subtype]
            group1 = subtype_data[subtype_data['Timepoint'] == timepoints[0]][marker]
            group2 = subtype_data[subtype_data['Timepoint'] == timepoints[1]][marker]
            
            # 使用 Mann-Whitney U test（对于非正态分布的数据）
            stat, p_val = stats.ttest_ind(group1, group2, equal_var=False)  # equal_var=False表示Welch's t-test，不假设方差相等
            p_values[subtype] = p_val
        
        # 计算padj（调整后的p值，Benjamini-Hochberg方法）
        p_vals = list(p_values.values())
        rej, padj = smm.multipletests(p_vals, alpha=0.05, method='fdr_bh')[:2]

        # 将padj值与每个subtype对应
        padj_dict = dict(zip(p_values.keys(), padj))

        # 标出显著性标签（根据padj而不是p值来显示显著性）
        for subtype, adjusted_p_val in padj_dict.items():
            if adjusted_p_val < 0.001:
                # 动态设置y坐标，避免与小提琴图重叠
                max_y = data_marker[marker].max()  # 获取当前marker的最大值
                if timepoints == ('V1Day0', 'V1Day1'):
                    # 对于V1Day0与V1Day1比较，标签偏左
                    x_position = subtypes.tolist().index(subtype) - 0.2
                elif timepoints == ('V2Day0', 'V2Day1'):
                    # 对于V2Day0与V2Day1比较，标签偏右
                    x_position = subtypes.tolist().index(subtype) + 0.2
                # 在图中添加显著性标记
                plt.text(x_position, max_y + 0.5, "***", horizontalalignment='center', 
                         verticalalignment='center', fontsize=8, color='black')
            elif adjusted_p_val < 0.01:
                # 动态设置y坐标，避免与小提琴图重叠
                max_y = data_marker[marker].max()  # 获取当前marker的最大值
                if timepoints == ('V1Day0', 'V1Day1'):
                    # 对于V1Day0与V1Day1比较，标签偏左
                    x_position = subtypes.tolist().index(subtype) - 0.2
                elif timepoints == ('V2Day0', 'V2Day1'):
                    # 对于V2Day0与V2Day1比较，标签偏右
                    x_position = subtypes.tolist().index(subtype) + 0.2
                # 在图中添加显著性标记
                plt.text(x_position, max_y + 0.5, "**", horizontalalignment='center', 
                         verticalalignment='center', fontsize=8, color='black')
            elif adjusted_p_val < 0.05:
                # 动态设置y坐标，避免与小提琴图重叠
                max_y = data_marker[marker].max()  # 获取当前marker的最大值
                if timepoints == ('V1Day0', 'V1Day1'):
                    # 对于V1Day0与V1Day1比较，标签偏左
                    x_position = subtypes.tolist().index(subtype) - 0.2
                elif timepoints == ('V2Day0', 'V2Day1'):
                    # 对于V2Day0与V2Day1比较，标签偏右
                    x_position = subtypes.tolist().index(subtype) + 0.2
                # 在图中添加显著性标记
                plt.text(x_position, max_y + 0.5, "*", horizontalalignment='center', 
                         verticalalignment='center', fontsize=8, color='black')

    # Step 14: 旋转x轴标签 45°并调整其位置
    plt.xticks(rotation=45, ha='right')
       # Step 15: 隐藏x轴标题
    plt.xlabel('')
    # Step 14: 保存每个小提琴图
    output_path = f'{output_dir}{marker}_violinplot.png'
    plt.savefig(output_path, bbox_inches='tight', dpi=300)

    # Step 15: 清空当前图形，准备下一张
    plt.close()

All markers are present in adata_all.var_names.


In [62]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats

# Step 1: 确保sub_markers的标记物存在于adata_all.var_names中
sub_markers = ['IgD_142Nd', 'IFN-a_144Nd', 'CXCL9_146Nd', 'CD86_147Sm', 'H3K27ac_148Nd', 
               'IL-1b_149Sm', '4E-BP1_150Nd', 'pS6_151Eu', 'CCR2_152Sm', 'TNF-a_153Eu', 
               'IL-4_154Sm', 'CASP-8_156Gd', 'CD38_159Tb', 'CD123_161Dy', 'Ki-67_162Dy', 
               'CXCL10_163Dy', 'Arginase-1_164Dy', 'IL-6_165Ho', 'MCP-1_166Er', 
               'CD69_167Er', 'STC1_168Er', 'IFN-g_172Yb', 'pSTAT1_175Lu', 'pSTAT3_176Yb','HLA-DR_171Yb']  # 你可以继续加入其他markers
sub_markers = ['CD86_147Sm', 'H3K27ac_148Nd','4E-BP1_150Nd', 'CASP-8_156Gd','CD123_161Dy' ,'IL-6_165Ho']
# 检查sub_markers中的每个标记物是否在adata_all.var_names中
missing_markers = [marker for marker in sub_markers if marker not in adata_all.var_names]
if missing_markers:
    print(f"The following markers are missing from adata_all.var_names: {missing_markers}")
else:
    print("All markers are present in adata_all.var_names.")

# Step 2: 获取标记物的列索引
sub_marker_indices = [adata_all.var_names.get_loc(marker) for marker in sub_markers]

# Step 3: 使用这些索引提取adata_all.X的相关列数据，并转换为DataFrame
data_long = adata_all.obs[['Timepoint', 'celltype']].join(pd.DataFrame(adata_all.X[:, sub_marker_indices], 
                                                                      columns=sub_markers, 
                                                                      index=adata_all.obs.index))

# Step 4: 设置保存路径
output_dir = '/public/users/xueyupeng/VZV/lvzhu/figures_celltype_violinplot_comp_adj/'

# Step 5: 获取celltype的所有唯一值，定义颜色
celltypes = data_long['celltype'].unique()
num_celltypes = len(celltypes)

# 使用tab20调色板，保证每个celltype都有一个唯一的颜色
celltype_palette = sns.color_palette("tab20", num_celltypes)
celltype_palette = dict(zip(celltypes, celltype_palette))

# Step 6: 获取Timepoint的所有唯一值并定义颜色
timepoints = data_long['Timepoint'].unique()
timepoint_palette = sns.color_palette("Set2", len(timepoints))  # 使用Set2调色板为时间点分配颜色
timepoint_palette = dict(zip(timepoints, timepoint_palette))

# Step 7: 遍历每个 marker 绘制小提琴图并保存
for marker in sub_markers:
    # Step 8: 过滤数据，选择当前 marker
    data_marker = data_long[['celltype', 'Timepoint', marker]]

    # Step 9: 为显著性检验创建两个比较组
    # 比较时间点：V1Day0与V1Day1、V2Day0与V2Day1
    comparison_groups = [
        (('V1Day0', 'V1Day1'), 'V1Day0 vs V1Day1'),
        (('V2Day0', 'V2Day1'), 'V2Day0 vs V2Day1')
    ]

    # Step 10: 绘制小提琴图
    plt.figure(figsize=(14, 6))
    sns.violinplot(x="celltype", y=marker, hue="Timepoint", data=data_marker, 
                   inner="quart", palette=timepoint_palette, split=True)
    
    # Step 11: 设置标题
    plt.title(f'{marker} Violin Plot by celltype and Timepoint')

    # Step 12: 调整图例
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))

    # Step 13: 进行显著性检验并标出显著性标签
    for (timepoints, comparison_label) in comparison_groups:
        # 过滤出指定时间点的数据
        timepoint_data = data_marker[data_marker['Timepoint'].isin(timepoints)]
        # 进行 Mann-Whitney U test 或 t-test 检验
        p_values = {}
        for celltype in celltypes:
            celltype_data = timepoint_data[timepoint_data['celltype'] == celltype]
            group1 = celltype_data[celltype_data['Timepoint'] == timepoints[0]][marker]
            group2 = celltype_data[celltype_data['Timepoint'] == timepoints[1]][marker]
            
            # 使用 Mann-Whitney U test（对于非正态分布的数据）
            stat, p_val = stats.ttest_ind(group1, group2, equal_var=False)  # equal_var=False表示Welch's t-test，不假设方差相等
            p_values[celltype] = p_val
        # 标出显著性标签（只显示 * 符号）
        for celltype, p_val in p_values.items():
            if p_val < 0.001:
                # 动态设置y坐标，避免与小提琴图重叠
                max_y = data_marker[marker].max()  # 获取当前marker的最大值
                if timepoints == ('V1Day0', 'V1Day1'):
                    # 对于V1Day0与V1Day1比较，标签偏左
                    x_position = celltypes.tolist().index(celltype) - 0.2
                elif timepoints == ('V2Day0', 'V2Day1'):
                    # 对于V2Day0与V2Day1比较，标签偏右
                    x_position = celltypes.tolist().index(celltype) + 0.2
                # 在图中添加显著性标记
                plt.text(x_position, max_y + 0.5, "***", horizontalalignment='center', 
                         verticalalignment='center', fontsize=8, color='black')
            elif  p_val < 0.01:
                # 动态设置y坐标，避免与小提琴图重叠
                max_y = data_marker[marker].max()  # 获取当前marker的最大值
                if timepoints == ('V1Day0', 'V1Day1'):
                    # 对于V1Day0与V1Day1比较，标签偏左
                    x_position = celltypes.tolist().index(celltype) - 0.2
                elif timepoints == ('V2Day0', 'V2Day1'):
                    # 对于V2Day0与V2Day1比较，标签偏右
                    x_position = celltypes.tolist().index(celltype) + 0.2
                # 在图中添加显著性标记
                plt.text(x_position, max_y + 0.5, "**", horizontalalignment='center', 
                         verticalalignment='center', fontsize=8, color='black')
            elif  p_val < 0.05:
                # 动态设置y坐标，避免与小提琴图重叠
                max_y = data_marker[marker].max()  # 获取当前marker的最大值
                if timepoints == ('V1Day0', 'V1Day1'):
                    # 对于V1Day0与V1Day1比较，标签偏左
                    x_position = celltypes.tolist().index(celltype) - 0.2
                elif timepoints == ('V2Day0', 'V2Day1'):
                    # 对于V2Day0与V2Day1比较，标签偏右
                    x_position = celltypes.tolist().index(celltype) + 0.2
                # 在图中添加显著性标记
                plt.text(x_position, max_y + 0.1, "*", horizontalalignment='center', 
                         verticalalignment='center', fontsize=8, color='black')
    # Step 14: 旋转x轴标签 45°并调整其位置
    plt.xticks(rotation=45, ha='right')
    # Step 15: 隐藏x轴标题
    plt.xlabel('')
    # Step 14: 保存每个小提琴图
    output_path = f'{output_dir}{marker}_violinplot.png'
    plt.savefig(output_path, bbox_inches='tight', dpi=300)

    # Step 15: 清空当前图形，准备下一张
    plt.close()

All markers are present in adata_all.var_names.


In [61]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.stats.multitest as smm

# Step 1: 确保sub_markers的标记物存在于adata_all.var_names中
sub_markers = ['IgD_142Nd', 'IFN-a_144Nd', 'CXCL9_146Nd', 'CD86_147Sm', 'H3K27ac_148Nd', 
               'IL-1b_149Sm', '4E-BP1_150Nd', 'pS6_151Eu', 'CCR2_152Sm', 'TNF-a_153Eu', 
               'IL-4_154Sm', 'CASP-8_156Gd', 'CD38_159Tb', 'CD123_161Dy', 'Ki-67_162Dy', 
               'CXCL10_163Dy', 'Arginase-1_164Dy', 'IL-6_165Ho', 'MCP-1_166Er', 
               'CD69_167Er', 'STC1_168Er', 'IFN-g_172Yb', 'pSTAT1_175Lu', 'pSTAT3_176Yb','HLA-DR_171Yb']  # 你可以继续加入其他markers
sub_markers = ['IFN-a_144Nd','CD86_147Sm', 'H3K27ac_148Nd','4E-BP1_150Nd', 'pS6_151Eu','CASP-8_156Gd','CD123_161Dy' ,'IL-6_165Ho']
# 检查sub_markers中的每个标记物是否在adata_all.var_names中
missing_markers = [marker for marker in sub_markers if marker not in adata_all.var_names]
if missing_markers:
    print(f"The following markers are missing from adata_all.var_names: {missing_markers}")
else:
    print("All markers are present in adata_all.var_names.")

# Step 2: 获取标记物的列索引
sub_marker_indices = [adata_all.var_names.get_loc(marker) for marker in sub_markers]

# Step 3: 使用这些索引提取adata_all.X的相关列数据，并转换为DataFrame
data_long = adata_all.obs[['Timepoint', 'celltype']].join(pd.DataFrame(adata_all.X[:, sub_marker_indices], 
                                                                      columns=sub_markers, 
                                                                      index=adata_all.obs.index))

# Step 4: 设置保存路径
output_dir = '/public/users/xueyupeng/VZV/lvzhu/figures_celltype_violinplot_comp_adj/'

# Step 5: 获取celltype的所有唯一值，定义颜色
celltypes = data_long['celltype'].unique()
num_celltypes = len(celltypes)

# 使用tab20调色板，保证每个celltype都有一个唯一的颜色
celltype_palette = sns.color_palette("tab20", num_celltypes)
celltype_palette = dict(zip(celltypes, celltype_palette))

# Step 6: 获取Timepoint的所有唯一值并定义颜色
timepoints = data_long['Timepoint'].unique()
timepoint_palette = sns.color_palette("Set2", len(timepoints))  # 使用Set2调色板为时间点分配颜色
timepoint_palette = dict(zip(timepoints, timepoint_palette))

# Step 7: 遍历每个 marker 绘制小提琴图并保存
for marker in sub_markers:
    # Step 8: 过滤数据，选择当前 marker
    data_marker = data_long[['celltype', 'Timepoint', marker]]

    # Step 9: 为显著性检验创建两个比较组
    # 比较时间点：V1Day0与V1Day1、V2Day0与V2Day1
    comparison_groups = [
        (('V1Day0', 'V1Day1'), 'V1Day0 vs V1Day1'),
        (('V2Day0', 'V2Day1'), 'V2Day0 vs V2Day1')
    ]

    # Step 10: 绘制小提琴图
    plt.figure(figsize=(14, 6))
    sns.violinplot(x="celltype", y=marker, hue="Timepoint", data=data_marker, 
                   inner="quart", palette=timepoint_palette, split=True)
    
    # Step 11: 设置标题
    plt.title(f'{marker} Violin Plot by celltype and Timepoint')

    # Step 12: 调整图例
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))

    # Step 13: 进行显著性检验并标出显著性标签
    for (timepoints, comparison_label) in comparison_groups:
        # 过滤出指定时间点的数据
        timepoint_data = data_marker[data_marker['Timepoint'].isin(timepoints)]
        # 进行 Mann-Whitney U test 或 t-test 检验
        p_values = {}
        for celltype in celltypes:
            celltype_data = timepoint_data[timepoint_data['celltype'] == celltype]
            group1 = celltype_data[celltype_data['Timepoint'] == timepoints[0]][marker]
            group2 = celltype_data[celltype_data['Timepoint'] == timepoints[1]][marker]
            
            # 使用 Mann-Whitney U test（对于非正态分布的数据）
            stat, p_val = stats.ttest_ind(group1, group2, equal_var=False)  # equal_var=False表示Welch's t-test，不假设方差相等
            p_values[celltype] = p_val
        
        # 计算padj（调整后的p值，Benjamini-Hochberg方法）
        p_vals = list(p_values.values())
        rej, padj = smm.multipletests(p_vals, alpha=0.05, method='fdr_bh')[:2]

        # 将padj值与每个celltype对应
        padj_dict = dict(zip(p_values.keys(), padj))

        # 标出显著性标签（根据padj而不是p值来显示显著性）
        for celltype, adjusted_p_val in padj_dict.items():
            if adjusted_p_val < 0.001:
                # 动态设置y坐标，避免与小提琴图重叠
                max_y = data_marker[marker].max()  # 获取当前marker的最大值
                if timepoints == ('V1Day0', 'V1Day1'):
                    # 对于V1Day0与V1Day1比较，标签偏左
                    x_position = celltypes.tolist().index(celltype) - 0.2
                elif timepoints == ('V2Day0', 'V2Day1'):
                    # 对于V2Day0与V2Day1比较，标签偏右
                    x_position = celltypes.tolist().index(celltype) + 0.2
                # 在图中添加显著性标记
                plt.text(x_position, max_y + 0.5, "***", horizontalalignment='center', 
                         verticalalignment='center', fontsize=8, color='black')
            elif adjusted_p_val < 0.01:
                # 动态设置y坐标，避免与小提琴图重叠
                max_y = data_marker[marker].max()  # 获取当前marker的最大值
                if timepoints == ('V1Day0', 'V1Day1'):
                    # 对于V1Day0与V1Day1比较，标签偏左
                    x_position = celltypes.tolist().index(celltype) - 0.2
                elif timepoints == ('V2Day0', 'V2Day1'):
                    # 对于V2Day0与V2Day1比较，标签偏右
                    x_position = celltypes.tolist().index(celltype) + 0.2
                # 在图中添加显著性标记
                plt.text(x_position, max_y + 0.5, "**", horizontalalignment='center', 
                         verticalalignment='center', fontsize=8, color='black')
            elif adjusted_p_val < 0.05:
                # 动态设置y坐标，避免与小提琴图重叠
                max_y = data_marker[marker].max()  # 获取当前marker的最大值
                if timepoints == ('V1Day0', 'V1Day1'):
                    # 对于V1Day0与V1Day1比较，标签偏左
                    x_position = celltypes.tolist().index(celltype) - 0.2
                elif timepoints == ('V2Day0', 'V2Day1'):
                    # 对于V2Day0与V2Day1比较，标签偏右
                    x_position = celltypes.tolist().index(celltype) + 0.2
                # 在图中添加显著性标记
                plt.text(x_position, max_y + 0.1, "*", horizontalalignment='center', 
                         verticalalignment='center', fontsize=8, color='black')

    # Step 14: 旋转x轴标签 45°并调整其位置
    plt.xticks(rotation=45, ha='right')
       # Step 15: 隐藏x轴标题
    plt.xlabel('')
    # Step 14: 保存每个小提琴图
    output_path = f'{output_dir}{marker}_violinplot.png'
    plt.savefig(output_path, bbox_inches='tight', dpi=300)

    # Step 15: 清空当前图形，准备下一张
    plt.close()

All markers are present in adata_all.var_names.


In [66]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats

# Step 1: 确保sub_markers的标记物存在于adata_all.var_names中
sub_markers = ['IgD_142Nd', 'IFN-a_144Nd', 'CXCL9_146Nd', 'CD86_147Sm', 'H3K27ac_148Nd', 
               'IL-1b_149Sm', '4E-BP1_150Nd', 'pS6_151Eu', 'CCR2_152Sm', 'TNF-a_153Eu', 
               'IL-4_154Sm', 'CASP-8_156Gd', 'CD38_159Tb', 'CD123_161Dy', 'Ki-67_162Dy', 
               'CXCL10_163Dy', 'Arginase-1_164Dy', 'IL-6_165Ho', 'MCP-1_166Er', 
               'CD69_167Er', 'STC1_168Er', 'IFN-g_172Yb', 'pSTAT1_175Lu', 'pSTAT3_176Yb','HLA-DR_171Yb']  # 你可以继续加入其他markers
sub_markers = ['IFN-a_144Nd','CD86_147Sm', 'H3K27ac_148Nd','4E-BP1_150Nd', 'pS6_151Eu','CASP-8_156Gd','CD123_161Dy' ,'IL-6_165Ho']
# 检查sub_markers中的每个标记物是否在adata_all.var_names中
missing_markers = [marker for marker in sub_markers if marker not in adata_all.var_names]
if missing_markers:
    print(f"The following markers are missing from adata_all.var_names: {missing_markers}")
else:
    print("All markers are present in adata_all.var_names.")

# Step 2: 获取标记物的列索引
sub_marker_indices = [adata_all.var_names.get_loc(marker) for marker in sub_markers]

# Step 3: 使用这些索引提取adata_all.X的相关列数据，并转换为DataFrame
data_long = adata_all.obs[['Timepoint', 'celltype']].join(pd.DataFrame(adata_all.X[:, sub_marker_indices], 
                                                                      columns=sub_markers, 
                                                                      index=adata_all.obs.index))

# Step 4: 设置保存路径
output_dir = '/public/users/xueyupeng/VZV/lvzhu/figures_celltype_violinplot_comp_adj/'

# Step 5: 获取celltype的所有唯一值，定义颜色
celltypes = data_long['celltype'].unique()
num_celltypes = len(celltypes)

# 使用tab20调色板，保证每个celltype都有一个唯一的颜色
celltype_palette = sns.color_palette("tab20", num_celltypes)
celltype_palette = dict(zip(celltypes, celltype_palette))

# Step 6: 获取Timepoint的所有唯一值并定义颜色
timepoints = data_long['Timepoint'].unique()
timepoint_palette = sns.color_palette("Set2", len(timepoints))  # 使用Set2调色板为时间点分配颜色
timepoint_palette = dict(zip(timepoints, timepoint_palette))

for marker in sub_markers:
    # Step 8: 过滤数据，选择当前 marker
    data_marker = data_long[['celltype', 'Timepoint', marker]]

    # Step 9: 为显著性检验创建两个比较组
    comparison_groups = [
        (('V1Day0', 'V1Day1'), 'V1Day0 vs V1Day1'),
        (('V2Day0', 'V2Day1'), 'V2Day0 vs V2Day1')
    ]

    # Step 10: 绘制小提琴图
    plt.figure(figsize=(14, 6))
    sns.violinplot(x="celltype", y=marker, hue="Timepoint", data=data_marker, 
                   inner="quart", palette=timepoint_palette, split=True)
    
    # Step 11: 设置标题
    plt.title(f'{marker} Violin Plot by celltype and Timepoint')

    # Step 12: 调整图例
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))

    # Step 13: 进行显著性检验并标出显著性标签
    for (timepoints, comparison_label) in comparison_groups:
        # 过滤出指定时间点的数据
        timepoint_data = data_marker[data_marker['Timepoint'].isin(timepoints)]
        # 进行 Mann-Whitney U test 或 t-test 检验
        p_values = {}
        for celltype in celltypes:
            celltype_data = timepoint_data[timepoint_data['celltype'] == celltype]
            group1 = celltype_data[celltype_data['Timepoint'] == timepoints[0]][marker]
            group2 = celltype_data[celltype_data['Timepoint'] == timepoints[1]][marker]
            
            # 使用 Mann-Whitney U test（对于非正态分布的数据）
            stat, p_val = stats.ttest_ind(group1, group2, equal_var=False)  # equal_var=False表示Welch's t-test，不假设方差相等
            p_values[celltype] = p_val
        
        # 计算padj（调整后的p值，Benjamini-Hochberg方法）
        p_vals = list(p_values.values())
        rej, padj = smm.multipletests(p_vals, alpha=0.05, method='fdr_bh')[:2]

        # 将padj值与每个celltype对应
        padj_dict = dict(zip(p_values.keys(), padj))

        # 标出显著性标签（根据padj而不是p值来显示显著性）
        for celltype, adjusted_p_val in padj_dict.items():
            # 动态设置y坐标，避免与小提琴图重叠
            max_y = data_marker[marker].max()  # 获取当前marker的最大值
            y_offset = 0.05  # 设置标签与图形顶部的距离

            # 根据调整后的p值确定标记
            if adjusted_p_val < 0.001:
                label = "***"
            elif adjusted_p_val < 0.01:
                label = "**"
            elif adjusted_p_val < 0.05:
                label = "*"
            else:
                continue  # 如果p值不显著则跳过
            
            # 计算x位置，确保标签偏左或偏右
            if timepoints == ('V1Day0', 'V1Day1'):
                x_position = celltypes.tolist().index(celltype) - 0.2
            elif timepoints == ('V2Day0', 'V2Day1'):
                x_position = celltypes.tolist().index(celltype) + 0.2

            # 获取当前图形的y轴范围，限制标签的最大y坐标
            ymin, ymax = plt.ylim()
            adjusted_max_y = min(max_y + y_offset, ymax)  # 确保标签不超过y轴最大值

            # 在图中添加显著性标记
            plt.text(x_position, adjusted_max_y, label, horizontalalignment='center', 
                     verticalalignment='center', fontsize=8, color='black')

    # Step 14: 旋转x轴标签 45°并调整其位置
    plt.xticks(rotation=45, ha='right')

    # Step 15: 隐藏x轴标题
    plt.xlabel('')
    
    # Step 16: 保存每个小提琴图
    output_path = f'{output_dir}{marker}_violinplot.png'
    plt.savefig(output_path, bbox_inches='tight', dpi=300)

    # Step 17: 清空当前图形，准备下一张
    plt.close()

All markers are present in adata_all.var_names.


In [64]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats

# Step 1: 确保sub_markers的标记物存在于adata_all.var_names中
sub_markers = ['IgD_142Nd', 'IFN-a_144Nd', 'CXCL9_146Nd', 'CD86_147Sm', 'H3K27ac_148Nd', 
               'IL-1b_149Sm', '4E-BP1_150Nd', 'pS6_151Eu', 'CCR2_152Sm', 'TNF-a_153Eu', 
               'IL-4_154Sm', 'CASP-8_156Gd', 'CD38_159Tb', 'CD123_161Dy', 'Ki-67_162Dy', 
               'CXCL10_163Dy', 'Arginase-1_164Dy', 'IL-6_165Ho', 'MCP-1_166Er', 
               'CD69_167Er', 'STC1_168Er', 'IFN-g_172Yb', 'pSTAT1_175Lu', 'pSTAT3_176Yb','HLA-DR_171Yb']  # 你可以继续加入其他markers
sub_markers = ['CD86_147Sm', 'H3K27ac_148Nd','4E-BP1_150Nd', 'CASP-8_156Gd','CD123_161Dy' ,'IL-6_165Ho']
# 检查sub_markers中的每个标记物是否在adata_all.var_names中
missing_markers = [marker for marker in sub_markers if marker not in adata_all.var_names]
if missing_markers:
    print(f"The following markers are missing from adata_all.var_names: {missing_markers}")
else:
    print("All markers are present in adata_all.var_names.")

# Step 2: 获取标记物的列索引
sub_marker_indices = [adata_all.var_names.get_loc(marker) for marker in sub_markers]

# Step 3: 使用这些索引提取adata_all.X的相关列数据，并转换为DataFrame
data_long = adata_all.obs[['Timepoint', 'subtype']].join(pd.DataFrame(adata_all.X[:, sub_marker_indices], 
                                                                      columns=sub_markers, 
                                                                      index=adata_all.obs.index))

# Step 4: 设置保存路径
output_dir = '/public/users/xueyupeng/VZV/lvzhu/figures_subtype_violinplot_comp_adj/'

# Step 5: 获取subtype的所有唯一值，定义颜色
subtypes = data_long['subtype'].unique()
num_subtypes = len(subtypes)

# 使用tab20调色板，保证每个subtype都有一个唯一的颜色
subtype_palette = sns.color_palette("tab20", num_subtypes)
subtype_palette = dict(zip(subtypes, subtype_palette))

# Step 6: 获取Timepoint的所有唯一值并定义颜色
timepoints = data_long['Timepoint'].unique()
timepoint_palette = sns.color_palette("Set2", len(timepoints))  # 使用Set2调色板为时间点分配颜色
timepoint_palette = dict(zip(timepoints, timepoint_palette))

for marker in sub_markers:
    # Step 8: 过滤数据，选择当前 marker
    data_marker = data_long[['subtype', 'Timepoint', marker]]

    # Step 9: 为显著性检验创建两个比较组
    comparison_groups = [
        (('V1Day0', 'V1Day1'), 'V1Day0 vs V1Day1'),
        (('V2Day0', 'V2Day1'), 'V2Day0 vs V2Day1')
    ]

    # Step 10: 绘制小提琴图
    plt.figure(figsize=(14, 6))
    sns.violinplot(x="subtype", y=marker, hue="Timepoint", data=data_marker, 
                   inner="quart", palette=timepoint_palette, split=True)
    
    # Step 11: 设置标题
    plt.title(f'{marker} Violin Plot by subtype and Timepoint')

    # Step 12: 调整图例
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))

    # Step 13: 进行显著性检验并标出显著性标签
    for (timepoints, comparison_label) in comparison_groups:
        # 过滤出指定时间点的数据
        timepoint_data = data_marker[data_marker['Timepoint'].isin(timepoints)]
        # 进行 Mann-Whitney U test 或 t-test 检验
        p_values = {}
        for subtype in subtypes:
            subtype_data = timepoint_data[timepoint_data['subtype'] == subtype]
            group1 = subtype_data[subtype_data['Timepoint'] == timepoints[0]][marker]
            group2 = subtype_data[subtype_data['Timepoint'] == timepoints[1]][marker]
            
            # 使用 Mann-Whitney U test（对于非正态分布的数据）
            stat, p_val = stats.ttest_ind(group1, group2, equal_var=False)  # equal_var=False表示Welch's t-test，不假设方差相等
            p_values[subtype] = p_val
        
        # 计算padj（调整后的p值，Benjamini-Hochberg方法）
        p_vals = list(p_values.values())
        rej, padj = smm.multipletests(p_vals, alpha=0.05, method='fdr_bh')[:2]

        # 将padj值与每个subtype对应
        padj_dict = dict(zip(p_values.keys(), padj))

        # 标出显著性标签（根据padj而不是p值来显示显著性）
        for subtype, adjusted_p_val in padj_dict.items():
            # 动态设置y坐标，避免与小提琴图重叠
            max_y = data_marker[marker].max()  # 获取当前marker的最大值
            y_offset = 0.05  # 设置标签与图形顶部的距离

            # 根据调整后的p值确定标记
            if adjusted_p_val < 0.001:
                label = "***"
            elif adjusted_p_val < 0.01:
                label = "**"
            elif adjusted_p_val < 0.05:
                label = "*"
            else:
                continue  # 如果p值不显著则跳过
            
            # 计算x位置，确保标签偏左或偏右
            if timepoints == ('V1Day0', 'V1Day1'):
                x_position = subtypes.tolist().index(subtype) - 0.2
            elif timepoints == ('V2Day0', 'V2Day1'):
                x_position = subtypes.tolist().index(subtype) + 0.2

            # 获取当前图形的y轴范围，限制标签的最大y坐标
            ymin, ymax = plt.ylim()
            adjusted_max_y = min(max_y + y_offset, ymax)  # 确保标签不超过y轴最大值

            # 在图中添加显著性标记
            plt.text(x_position, adjusted_max_y, label, horizontalalignment='center', 
                     verticalalignment='center', fontsize=8, color='black')

    # Step 14: 旋转x轴标签 45°并调整其位置
    plt.xticks(rotation=45, ha='right')

    # Step 15: 隐藏x轴标题
    plt.xlabel('')
    
    # Step 16: 保存每个小提琴图
    output_path = f'{output_dir}{marker}_violinplot.png'
    plt.savefig(output_path, bbox_inches='tight', dpi=300)

    # Step 17: 清空当前图形，准备下一张
    plt.close()

All markers are present in adata_all.var_names.
